In [1]:
import csv
import os
import random
from datetime import datetime, timedelta

### Output Folder Setup

In [2]:
OUTPUT_DIR = "data/generated"

# Create folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory ready: {OUTPUT_DIR}")


Output directory ready: data/generated


### Generating Teams

In [3]:
def generate_teams():
    teams = [
        {"team_id": 1, "team_name": "Data Engineering", "team_email": "de@example.com"},
        {"team_id": 2, "team_name": "Analytics", "team_email": "analytics@example.com"},
        {"team_id": 3, "team_name": "ML Platform", "team_email": "ml@example.com"},
    ]

    file_path = os.path.join(OUTPUT_DIR, "teams.csv")

    with open(file_path, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=teams[0].keys())
        writer.writeheader()
        writer.writerows(teams)

    print(f"Generated {file_path}")

In [4]:
generate_teams()

Generated data/generated/teams.csv


### Generating Pipelines

In [5]:
def generate_pipelines():
    pipelines = []

    pipeline_names = [
        "daily_sales_pipeline",
        "user_activity_pipeline",
        "inventory_pipeline",
        "payments_pipeline",
        "marketing_pipeline",
        "fraud_detection_pipeline"
    ]

    pipeline_id = 1

    for name in pipeline_names:
        pipelines.append({
            "pipeline_id": pipeline_id,
            "pipeline_name": name,
            "team_id": random.randint(1, 3),  # assign to random team
            "schedule_type": random.choice(["daily", "hourly"]),
            "criticality": random.choice(["high", "medium", "low"]),
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        pipeline_id += 1

    file_path = os.path.join(OUTPUT_DIR, "pipelines.csv")

    with open(file_path, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=pipelines[0].keys())
        writer.writeheader()
        writer.writerows(pipelines)

    print(f"Generated {file_path}")

    return pipelines
    

In [6]:
pipelines = generate_pipelines()

Generated data/generated/pipelines.csv


### Generating Tasks

In [7]:
def generate_tasks(pipelines):
    tasks = []
    task_id = 1

    task_templates = [
        ("extract_data", "extract", 1),
        ("transform_data", "transform", 2),
        ("load_data", "load", 3)
    ]

    for pipeline in pipelines:
        for task_name, task_type, task_order in task_templates:
            tasks.append({
                "task_id": task_id,
                "pipeline_id": pipeline["pipeline_id"],
                "task_name": task_name,
                "task_type": task_type,
                "task_order": task_order,
                "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            task_id += 1

    file_path = os.path.join(OUTPUT_DIR, "tasks.csv")

    with open(file_path, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=tasks[0].keys())
        writer.writeheader()
        writer.writerows(tasks)

    print(f"Generated {file_path}")

    return tasks

In [8]:
tasks = generate_tasks(pipelines)

Generated data/generated/tasks.csv


### Generating Data Assets

In [9]:
def generate_data_assets(pipelines):
    data_assets = []
    asset_id = 1

    for pipeline in pipelines:
        pipeline_name = pipeline["pipeline_name"].replace("_pipeline", "")
        data_assets.append({
            "asset_id": asset_id,
            "asset_name": f"{pipeline_name}_table",
            "pipeline_id": pipeline["pipeline_id"],
            "asset_type": "table",
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        asset_id += 1

    file_path = os.path.join(OUTPUT_DIR, "data_assets.csv")

    with open(file_path, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=data_assets[0].keys())
        writer.writeheader()
        writer.writerows(data_assets)

    print(f"Generated {file_path}")

    return data_assets

In [10]:
data_assets = generate_data_assets(pipelines)

Generated data/generated/data_assets.csv


### Helper map for tasks by pipeline

In [11]:
def build_tasks_by_pipeline(tasks):
    tasks_by_pipeline = {}

    for task in tasks:
        pipeline_id = task["pipeline_id"]
        tasks_by_pipeline.setdefault(pipeline_id, []).append(task)

    # sort tasks by task_order
    for pipeline_id in tasks_by_pipeline:
        tasks_by_pipeline[pipeline_id] = sorted(
            tasks_by_pipeline[pipeline_id],
            key=lambda x: x["task_order"]
        )

    return tasks_by_pipeline

In [12]:
tasks_by_pipeline = build_tasks_by_pipeline(tasks)
tasks_by_pipeline[1]

[{'task_id': 1,
  'pipeline_id': 1,
  'task_name': 'extract_data',
  'task_type': 'extract',
  'task_order': 1,
  'created_at': '2026-04-12 22:07:05'},
 {'task_id': 2,
  'pipeline_id': 1,
  'task_name': 'transform_data',
  'task_type': 'transform',
  'task_order': 2,
  'created_at': '2026-04-12 22:07:05'},
 {'task_id': 3,
  'pipeline_id': 1,
  'task_name': 'load_data',
  'task_type': 'load',
  'task_order': 3,
  'created_at': '2026-04-12 22:07:05'}]

### Generating the pipeline events

In [13]:
def generate_pipeline_events(pipelines, tasks_by_pipeline, days=30):
    events = []
    event_id = 1
    current_time = datetime.now()

    # duration ranges in minutes per task type
    task_durations = {
        "extract": (5, 10),
        "transform": (10, 20),
        "load": (3, 8)
    }

    for pipeline in pipelines:
        pipeline_id = pipeline["pipeline_id"]

        for day in range(days):
            date = current_time - timedelta(days=day)
            runs_per_day = random.randint(1, 2)

            for _ in range(runs_per_day):

                # pipeline starts at a random time that day
                run_start = date.replace(
                    hour=random.randint(0, 22),
                    minute=random.randint(0, 59),
                    second=0,
                    microsecond=0
                )

                # pipeline started event
                events.append({
                    "event_id": event_id,
                    "pipeline_id": pipeline_id,
                    "task_id": None,
                    "event_type": "pipeline_started",
                    "status": "running",
                    "event_timestamp": run_start.strftime("%Y-%m-%d %H:%M:%S"),
                    "duration_sec": None,
                    "records_in": None,
                    "records_out": None,
                    "error_message": None,
                    "created_at": run_start.strftime("%Y-%m-%d %H:%M:%S")
                })
                event_id += 1

                pipeline_failed = False
                task_start = run_start

                # iterate tasks sequentially
                for task in tasks_by_pipeline[pipeline_id]:
                    task_id = task["task_id"]
                    task_type = task["task_type"]

                    # task starts after previous task ends
                    task_start = task_start + timedelta(minutes=random.randint(1, 3))

                    # task started event
                    events.append({
                        "event_id": event_id,
                        "pipeline_id": pipeline_id,
                        "task_id": task_id,
                        "event_type": "task_started",
                        "status": "running",
                        "event_timestamp": task_start.strftime("%Y-%m-%d %H:%M:%S"),
                        "duration_sec": None,
                        "records_in": None,
                        "records_out": None,
                        "error_message": None,
                        "created_at": task_start.strftime("%Y-%m-%d %H:%M:%S")
                    })
                    event_id += 1

                    # calculate task duration
                    min_dur, max_dur = task_durations.get(task_type, (5, 15))
                    duration_min = random.randint(min_dur, max_dur)
                    task_end = task_start + timedelta(minutes=duration_min)

                    # 10% chance of failure
                    if random.random() < 0.1:
                        pipeline_failed = True

                        # task failed event
                        events.append({
                            "event_id": event_id,
                            "pipeline_id": pipeline_id,
                            "task_id": task_id,
                            "event_type": "task_failed",
                            "status": "failed",
                            "event_timestamp": task_end.strftime("%Y-%m-%d %H:%M:%S"),
                            "duration_sec": duration_min * 60,
                            "records_in": random.randint(1000, 50000),
                            "records_out": None,
                            "error_message": random.choice([
                                "null values encountered",
                                "connection timeout",
                                "schema mismatch",
                                "out of memory",
                                "source data unavailable"
                            ]),
                            "created_at": task_end.strftime("%Y-%m-%d %H:%M:%S")
                        })
                        event_id += 1

                        # retry event
                        retry_start = task_end + timedelta(minutes=2)
                        events.append({
                            "event_id": event_id,
                            "pipeline_id": pipeline_id,
                            "task_id": task_id,
                            "event_type": "task_retry",
                            "status": "retrying",
                            "event_timestamp": retry_start.strftime("%Y-%m-%d %H:%M:%S"),
                            "duration_sec": None,
                            "records_in": None,
                            "records_out": None,
                            "error_message": None,
                            "created_at": retry_start.strftime("%Y-%m-%d %H:%M:%S")
                        })
                        event_id += 1

                        # update task_start for next task after retry
                        task_start = retry_start + timedelta(minutes=duration_min)

                    else:
                        # task succeeded
                        records_in = random.randint(1000, 50000)
                        events.append({
                            "event_id": event_id,
                            "pipeline_id": pipeline_id,
                            "task_id": task_id,
                            "event_type": "task_succeeded",
                            "status": "success",
                            "event_timestamp": task_end.strftime("%Y-%m-%d %H:%M:%S"),
                            "duration_sec": duration_min * 60,
                            "records_in": records_in,
                            "records_out": random.randint(int(records_in * 0.8), records_in),
                            "error_message": None,
                            "created_at": task_end.strftime("%Y-%m-%d %H:%M:%S")
                        })
                        event_id += 1

                        # update task_start for next task
                        task_start = task_end

                # pipeline completed event
                pipeline_end = task_start + timedelta(minutes=1)
                events.append({
                    "event_id": event_id,
                    "pipeline_id": pipeline_id,
                    "task_id": None,
                    "event_type": "pipeline_completed",
                    "status": "success" if not pipeline_failed else "completed_with_issues",
                    "event_timestamp": pipeline_end.strftime("%Y-%m-%d %H:%M:%S"),
                    "duration_sec": int((pipeline_end - run_start).total_seconds()),
                    "records_in": None,
                    "records_out": None,
                    "error_message": None,
                    "created_at": pipeline_end.strftime("%Y-%m-%d %H:%M:%S")
                })
                event_id += 1

    print(f"Generated {len(events)} events")
    return events

In [14]:
events = generate_pipeline_events(pipelines, tasks_by_pipeline, days=30)
len(events)

Generated 2185 events


2185

In [15]:
events[:10]

[{'event_id': 1,
  'pipeline_id': 1,
  'task_id': None,
  'event_type': 'pipeline_started',
  'status': 'running',
  'event_timestamp': '2026-04-12 12:18:00',
  'duration_sec': None,
  'records_in': None,
  'records_out': None,
  'error_message': None,
  'created_at': '2026-04-12 12:18:00'},
 {'event_id': 2,
  'pipeline_id': 1,
  'task_id': 1,
  'event_type': 'task_started',
  'status': 'running',
  'event_timestamp': '2026-04-12 12:20:00',
  'duration_sec': None,
  'records_in': None,
  'records_out': None,
  'error_message': None,
  'created_at': '2026-04-12 12:20:00'},
 {'event_id': 3,
  'pipeline_id': 1,
  'task_id': 1,
  'event_type': 'task_succeeded',
  'status': 'success',
  'event_timestamp': '2026-04-12 12:29:00',
  'duration_sec': 540,
  'records_in': 18040,
  'records_out': 15578,
  'error_message': None,
  'created_at': '2026-04-12 12:29:00'},
 {'event_id': 4,
  'pipeline_id': 1,
  'task_id': 2,
  'event_type': 'task_started',
  'status': 'running',
  'event_timestamp': '20

In [17]:
def save_events_to_csv(events):
    file_path = os.path.join(OUTPUT_DIR, "raw_pipeline_events.csv")

    with open(file_path, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=events[0].keys())
        writer.writeheader()
        writer.writerows(events)

    print(f"Generated {file_path} with {len(events)} rows")

save_events_to_csv(events)

Generated data/generated/raw_pipeline_events.csv with 2185 rows


In [18]:
def generate_slas(pipelines):
    slas = []
    sla_id = 1

    for pipeline in pipelines:
        slas.append({
            "sla_id": sla_id,
            "pipeline_id": pipeline["pipeline_id"],
            "expected_completion_time": random.choice(["06:00:00", "08:00:00", "12:00:00", "18:00:00"]),
            "max_runtime_sec": random.choice([3600, 7200, 10800]),
            "freshness_threshold_min": random.choice([60, 120, 240]),
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        sla_id += 1

    file_path = os.path.join(OUTPUT_DIR, "slas.csv")

    with open(file_path, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=slas[0].keys())
        writer.writeheader()
        writer.writerows(slas)

    print(f"Generated {file_path} with {len(slas)} rows")
    return slas

slas = generate_slas(pipelines)

Generated data/generated/slas.csv with 6 rows


In [19]:
def generate_schema_change_events(pipelines, data_assets, days=30):
    schema_changes = []
    change_id = 1
    current_time = datetime.now()

    change_types = ["column_added", "column_removed", "type_changed", "column_renamed"]
    initiators = ["data_engineer", "automated_migration", "dbt_run", "platform_team"]

    for pipeline in pipelines:
        # each pipeline gets 2-4 schema changes over the period
        num_changes = random.randint(2, 4)

        # find the asset for this pipeline
        asset = next((a for a in data_assets if a["pipeline_id"] == pipeline["pipeline_id"]), None)
        if not asset:
            continue

        for _ in range(num_changes):
            change_day = random.randint(1, days)
            change_time = current_time - timedelta(days=change_day)

            schema_changes.append({
                "change_id": change_id,
                "pipeline_id": pipeline["pipeline_id"],
                "asset_id": asset["asset_id"],
                "change_type": random.choice(change_types),
                "old_schema": "schema_v1",
                "new_schema": "schema_v2",
                "initiated_by": random.choice(initiators),
                "change_ts": change_time.strftime("%Y-%m-%d %H:%M:%S"),
                "created_at": change_time.strftime("%Y-%m-%d %H:%M:%S")
            })
            change_id += 1

    file_path = os.path.join(OUTPUT_DIR, "schema_change_events.csv")

    with open(file_path, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=schema_changes[0].keys())
        writer.writeheader()
        writer.writerows(schema_changes)

    print(f"Generated {file_path} with {len(schema_changes)} rows")
    return schema_changes

schema_changes = generate_schema_change_events(pipelines, data_assets, days=30)

Generated data/generated/schema_change_events.csv with 18 rows
